In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement
#%run ./load_data ----- A decommenter pour lancer le notebook separement
#%run ./transform_data ----- A decommenter pour lancer le notebook separement

In [0]:
#%run ./env

In [0]:
#%run ./python_libraries 

In [0]:
#%run ../delta_function 

In [0]:
#%run ./load_data 

In [0]:
#%run ./transform_data 

## Construction dim_batch
Une ligne = un batch. Cle primaire = batch_id.

In [0]:
# Pas de filtre sur `deleted` ici : le mode update de handle_table_update ne fait
# que des inserts et des updates, jamais de delete. Un batch ecarte a la source
# disparait du snapshot, sa ligne cible n'est plus jamais revisitee par le MERGE
# et resterait figee avec deleted = False -- l'inverse de la verite. En laissant
# passer les supprimes, `deleted` fait partie des additional_columns_to_check et
# bascule a True au run suivant. Le filtre deleted = False se fait cote Power BI.
dim_batch = (
    batches
    .select(
        F.col("id_batch").alias("batch_id"),
        F.col("batch_number"),
        F.col("requirement_specifications"),
        F.col("variety"),
        F.col("fabrication_order_number"),
        F.col("mes_number"),
        F.col("planned_date"),
        F.col("harvest"),
        F.col("production_line"),
        F.col("production_type"),
        F.col("status"),
        F.col("deleted"),
        F.col("batch_cycle"),
        F.col("planned_time"),
        F.col("planned_datetime"),
        F.col("created_at"),
        F.col("updated_at"),
        F.col("deleted_at")
    )
)

# production_line reste en FK brute (int) : la relation vers dim_site
# (id_plant_production_line) se fait cote Power BI, pas de jointure ici.

# max_end_date_kiln_unload : attribut du batch, calcule une fois, vraie Date.
# Regroupement par batch_id uniquement (pas prd_line) : evite les doublons
# lies a d'eventuelles variations de prd_line pour un meme batch.
max_end_date_kiln_unload = (
    localization_events_union
    .filter(
        (F.col("localization_event") == "kiln_unload") &
        (F.col("batch_id").isNotNull())
    )
    .groupBy("batch_id")
    .agg(F.max("end").alias("max_end_date_kiln_unload_brute"))
    .withColumn("max_end_date_kiln_unload", F.to_date(F.col("max_end_date_kiln_unload_brute")))
    .select("batch_id", "max_end_date_kiln_unload")
)

dim_batch = (
    dim_batch.alias("a")
    .join(
        max_end_date_kiln_unload.alias("b"),
        F.col("a.batch_id") == F.col("b.batch_id"),
        "left"
    )
    .select(
        F.col("a.batch_id"),
        F.col("a.batch_number"),
        F.col("a.requirement_specifications"),
        F.col("a.variety"),
        F.col("a.fabrication_order_number"),
        F.col("a.mes_number"),
        F.col("a.planned_date"),
        F.col("a.harvest"),
        F.col("a.production_line"),
        F.col("a.production_type"),
        F.col("a.status"),
        F.col("a.deleted"),
        F.col("a.batch_cycle"),
        F.col("a.planned_time"),
        F.col("a.planned_datetime"),
        F.col("a.created_at"),
        F.col("a.updated_at"),
        F.col("a.deleted_at"),
        F.col("b.max_end_date_kiln_unload")
    )
)

## Libelles specification et variete
Les FK `requirement_specifications` et `variety` sont resolues ici, dans dim_batch,
plutot que laissees a des dimensions separees cote Power BI. Une ligne = un batch,
le nombre de lignes est inchange.

In [0]:
# Libelles resolus ici plutot que cote Power BI : regrouper une matrice sur une
# colonne de dim_requirement_specification ET une colonne de dim_variety fait
# echouer le visuel ("les donnees depassent la capacite"), ces deux dimensions
# etant reliees a dim_batch mais pas entre elles.
#
# Filtre `deleted == False` cote dimensions : seuls les libelles actifs sont
# repris. Contrairement a batches, ces deux tables ne sont que des tables de
# correspondance dans un LEFT JOIN, jamais des lignes de dim_batch : le filtre
# est reevalue a chaque run, il ne fige rien. Un batch rattache a une
# specification ou une variete supprimee depuis remonte avec un libelle vide.

specification_labels = (
    requirement_specifications
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_requirement_specification"),
        F.col("name").alias("specification_name")
    )
)

variety_labels = (
    goods_varieties
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_good_variety"),
        F.col("code").alias("variety_code")
    )
)

dim_batch = (
    dim_batch.alias("a")
    .join(
        specification_labels.alias("b"),
        F.col("a.requirement_specifications") == F.col("b.id_requirement_specification"),
        "left"
    )
    .select(
        F.col("a.batch_id"),
        F.col("a.batch_number"),
        F.col("a.requirement_specifications"),
        F.col("a.variety"),
        F.col("a.fabrication_order_number"),
        F.col("a.mes_number"),
        F.col("a.planned_date"),
        F.col("a.harvest"),
        F.col("a.production_line"),
        F.col("a.production_type"),
        F.col("a.status"),
        F.col("a.deleted"),
        F.col("a.batch_cycle"),
        F.col("a.planned_time"),
        F.col("a.planned_datetime"),
        F.col("a.created_at"),
        F.col("a.updated_at"),
        F.col("a.deleted_at"),
        F.col("a.max_end_date_kiln_unload"),
        F.col("b.specification_name")
    )
)

dim_batch = (
    dim_batch.alias("a")
    .join(
        variety_labels.alias("b"),
        F.col("a.variety") == F.col("b.id_good_variety"),
        "left"
    )
    .select(
        F.col("a.batch_id"),
        F.col("a.batch_number"),
        F.col("a.requirement_specifications"),
        F.col("a.variety"),
        F.col("a.fabrication_order_number"),
        F.col("a.mes_number"),
        F.col("a.planned_date"),
        F.col("a.harvest"),
        F.col("a.production_line"),
        F.col("a.production_type"),
        F.col("a.status"),
        F.col("a.deleted"),
        F.col("a.batch_cycle"),
        F.col("a.planned_time"),
        F.col("a.planned_datetime"),
        F.col("a.created_at"),
        F.col("a.updated_at"),
        F.col("a.deleted_at"),
        F.col("a.max_end_date_kiln_unload"),
        F.col("a.specification_name"),
        F.col("b.variety_code")
    )
)


## Ecriture Delta

In [0]:
current_process = "dim_batch"
target_dim_batch = current_catalog + "." + current_schema + "." + current_process
print(target_dim_batch)

In [0]:
all_columns = dim_batch.columns
primary_key = ['batch_id']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    dim_batch,
    target_dim_batch,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)